# Experiment 7: Syntactic Analysis System: Constituency Parsing & Dependency Parsing

### Problem Statement:
Design a syntactic analysis system that performs constituency parsing and dependency parsing to discover grammatical relationships among words in complex sentences.


In [1]:
import nltk
import spacy
from spacy import displacy
import pandas as pd
from IPython.display import display, HTML

nlp = spacy.load("en_core_web_sm")
print("Syntactic parsing tools and models initialized.")


Syntactic parsing tools and models initialized.


In [2]:
grammar = nltk.CFG.fromstring('''
    S -> NP VP
    PP -> P NP
    NP -> Det N | Det N PP | 'I'
    VP -> V NP | VP PP
    Det -> 'the' | 'a'
    N -> 'man' | 'telescope' | 'park' | 'dog'
    V -> 'saw' | 'spotted'
    P -> 'with' | 'in'
''')

ambiguous_sentence = "I saw the man with a telescope".split()
parser = nltk.ChartParser(grammar)
parse_trees = list(parser.parse(ambiguous_sentence))

print(f"Grammar non-terminals: {len(grammar.productions())} production rules")
print(f"Total Parse Trees Found for ambiguous sentence: {len(parse_trees)}\n")

for i, tree in enumerate(parse_trees, 1):
    print(f"--- PARSE TREE {i} ---")
    tree.pretty_print()
    print(f"Bracketed Representation {i}:\n{tree}\n")


Grammar non-terminals: 17 production rules
Total Parse Trees Found for ambiguous sentence: 2

--- PARSE TREE 1 ---
     S                                    
  ___|___________                          
 |               VP                       
 |        _______|________                 
 |       VP               PP              
 |    ___|___         ____|___             
 |   |       NP      |        NP          
 |   |    ___|___    |     ___|______      
 NP  V  Det      N   P   Det         N    
 |   |   |       |   |    |          |     
 I  saw the     man with  a      telescope

Bracketed Representation 1:
(S
  (NP I)
  (VP
    (VP (V saw) (NP (Det the) (N man)))
    (PP (P with) (NP (Det a) (N telescope)))))

--- PARSE TREE 2 ---
     S                                
  ___|_______                          
 |           VP                       
 |    _______|___                      
 |   |           NP                   
 |   |    _______|____                 
 |   |   |   |

In [3]:
complex_sentence = "The artificial intelligence system accurately predicted stock market trends during volatile periods."
doc_dep = nlp(complex_sentence)

dep_records = []
for token in doc_dep:
    dep_records.append({
        "Token": token.text,
        "Dependency Tag": token.dep_,
        "Dependency Meaning": spacy.explain(token.dep_),
        "Head (Governor)": token.head.text,
        "Head POS": token.head.pos_,
        "Children": [child.text for child in token.children]
    })

df_dep = pd.DataFrame(dep_records)
print("--- Dependency Parse Relations Table ---")
print(df_dep.to_string(index=False))


--- Dependency Parse Relations Table ---
       Token Dependency Tag     Dependency Meaning Head (Governor) Head POS                                Children
         The            det             determiner          system     NOUN                                      []
  artificial           amod    adjectival modifier          system     NOUN                                      []
intelligence       compound               compound          system     NOUN                                      []
      system          nsubj        nominal subject       predicted     VERB         [The, artificial, intelligence]
  accurately         advmod     adverbial modifier       predicted     VERB                                      []
   predicted           ROOT                   root       predicted     VERB [system, accurately, trends, during, .]
       stock       compound               compound          market     NOUN                                      []
      market       compound    

In [4]:
# Render dependency visualization to HTML
dep_svg = displacy.render(doc_dep, style="dep", jupyter=False, options={"distance": 105, "compact": False})
display(HTML(f"<div style='max-width: 100%; overflow-x: auto;'>{dep_svg}</div>"))


In [5]:
def extract_svo_triplets(doc):
    triplets = []
    
    for token in doc:
        # Locate main verbs
        if token.pos_ in ("VERB", "AUX"):
            subject = None
            direct_object = None
            
            # Find subject and object among token children
            for child in token.children:
                if child.dep_ in ("nsubj", "nsubjpass"):
                    # Expand compound nouns
                    subj_prefix = "".join([c.text + " " for c in child.children if c.dep_ in ("compound", "amod")])
                    subject = (subj_prefix + child.text).strip()
                    
                elif child.dep_ in ("dobj", "obj", "attr"):
                    # Expand compound nouns
                    obj_prefix = "".join([c.text + " " for c in child.children if c.dep_ in ("compound", "amod")])
                    direct_object = (obj_prefix + child.text).strip()
                    
            if subject and direct_object:
                triplets.append({
                    "Subject": subject,
                    "Verb (Predicate)": token.lemma_,
                    "Object": direct_object
                })
                
    return triplets

test_passages = [
    "Google acquired DeepMind in London.",
    "OpenAI develops large language models.",
    "The autonomous vehicle detected pedestrian hazards quickly.",
    "Researchers published comprehensive empirical benchmarks."
]

extracted_knowledge = []
for passage in test_passages:
    d = nlp(passage)
    trips = extract_svo_triplets(d)
    for t in trips:
        t["Sentence"] = passage
        extracted_knowledge.append(t)

df_svo = pd.DataFrame(extracted_knowledge)[["Sentence", "Subject", "Verb (Predicate)", "Object"]]
print("--- Extracted Subject-Verb-Object (SVO) Knowledge Triplets ---")
print(df_svo.to_string(index=False))


--- Extracted Subject-Verb-Object (SVO) Knowledge Triplets ---
                                                   Sentence            Subject Verb (Predicate)                             Object
                        Google acquired DeepMind in London.             Google          acquire                           DeepMind
                     OpenAI develops large language models.             OpenAI          develop              large language models
The autonomous vehicle detected pedestrian hazards quickly. autonomous vehicle           detect                 pedestrian hazards
  Researchers published comprehensive empirical benchmarks.        Researchers          publish comprehensive empirical benchmarks
